In [4]:
import sdmetrics
import pandas as pd
import os
import json
from sdv.metadata import SingleTableMetadata
from sdmetrics.reports.single_table import DiagnosticReport, QualityReport
from sdmetrics.single_table import NewRowSynthesis
from sdmetrics.visualization import get_column_plot, get_column_pair_plot
from sdmetrics.single_column import TVComplement
print(sdmetrics.__version__)

0.12.1


Generate metadata

In [ ]:
# Generate metadata for ICU DKA dataset - only needs to be run once
real = pd.read_csv("C:/Users/maria/Code/master/xai-synthetic-health/dp_cgans/resources/icu_dka_dataset_20250723.csv")

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=real)

metadata.save_to_json("icu_dka_metadata.json")


In [5]:
def get_submetadata(metadata, columns):
    submetadata = {'columns': {}, 'primary_key': metadata.get('primary_key', None), 'METADATA_SPEC_VERSION': metadata['METADATA_SPEC_VERSION'], 'primary_key': metadata['primary_key'],}
    for col in columns:
        submetadata['columns'][col] = metadata['columns'][col]
    return submetadata

#### Evaluation

In [6]:
def evaluate_synthetic(
    exp_name: str,
    real_data: pd.DataFrame,
    syn_data: pd.DataFrame,
    metadata: dict,
    out_dir: str = "results_sdmetrics",
    columns: list = None,
):
    """
    Run DiagnosticReport + QualityReport on (real, synthetic) 
    and save all summaries/details for this experiment.
    """

    os.makedirs(out_dir, exist_ok=True)
    exp_dir = os.path.join(out_dir, exp_name)
    os.makedirs(exp_dir, exist_ok=True)

    # --- Save basic info about the experiment ---
    meta_path = os.path.join(exp_dir, "metadata.json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    # --- 1. Diagnostic Report ---
    diagnostic = DiagnosticReport()
    diagnostic.generate(real_data, syn_data, metadata)

    for prop in ["Coverage", "Boundary", "Synthesis"]:
        details = diagnostic.get_details(prop)
        details.to_csv(os.path.join(exp_dir, f"diagnostic_{prop.lower()}_details.csv"), index=False)

    # --- 2. Quality Report ---
    quality = QualityReport()
    quality.generate(real_data, syn_data, metadata)

    for prop in ["Column Shapes", "Column Pair Trends"]:
        try:
            details = quality.get_details(prop)
            details.to_csv(os.path.join(exp_dir, f"quality_{prop.replace(' ', '_').lower()}_details.csv"), index=False)
        except ValueError:
            pass

    
    score = NewRowSynthesis.compute(
        real_data=real_data,
        synthetic_data=syn_data,
        metadata=metadata
    )
    print(f"New Row Synthesis Score: {score}")
    return diagnostic, quality, exp_dir


In [7]:
def save_column_plot( exp_dir, real_data, syn_data, column_name):
    fig = get_column_plot(
        real_data=real_data,
        synthetic_data=syn_data,
        column_name=column_name,
    )
    fig.write_html(os.path.join(exp_dir, f"column_plot_{column_name}.html"))
    # fig.write_image(os.path.join(exp_dir, f"column_plot_{column_name}.png"))
    fig.show()

In [8]:
def save_column_pair_plot( exp_dir, real_data, syn_data, columns_name):
    fig = get_column_pair_plot(
        real_data=real_data,
        synthetic_data=syn_data,
        column_names=columns_name,
    )
    fig.write_html(os.path.join(exp_dir, f"column_plot_{columns_name[0]}_{columns_name[1]}.html"))
    fig.show()

# Diagnostic and Quality

## SHAP based features

In [9]:
dataset_path = "C:/Users/maria/OneDrive - Maastricht University/Maria Diez Perez/datasets/"
real_data = pd.read_csv(os.path.join(dataset_path, 'shap_30_important_features_icu_dka_dataset.csv'))

### Real VS Baseline

In [10]:
syn_data_baseline = pd.read_csv("../dp_cgans/tests/results/syn_data_shap_baseline.csv")
syn_data_baseline= syn_data_baseline.drop(syn_data_baseline.columns[0], axis=1)
metadata = get_submetadata(SingleTableMetadata.load_from_json("icu_dka_metadata.json").to_dict(), real_data.columns)

In [11]:
diagnostic_report, quality_report, folder = evaluate_synthetic(
    exp_name="baseline_shap_30features",
    real_data=real_data,
    syn_data=syn_data_baseline,
    metadata=metadata,
    out_dir="results_sdmetrics"
)

Generating report ...
(3/3) Evaluating Synthesis: : 100%|██████████| 1/1 [01:14<00:00, 74.15s/it]

Diagnostic Results:

SUCCESS:
✓ The synthetic data covers over 90% of the numerical ranges present in the real data
✓ The synthetic data follows over 90% of the min/max boundaries set by the real data
✓ Over 90% of the synthetic rows are not copies of the real data

! The synthetic data is missing more than 10% of the categories present in the real data
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 465/465 [00:06<00:00, 77.18it/s] 

Overall Quality Score: 88.82%

Properties:
- Column Shapes: 88.49%
- Column Pair Trends: 89.15%
New Row Synthesis Score: 1.0


In [12]:
for col in ["age", "lactate_max", "sofa"]:
    try:
        save_column_plot(folder, real_data, syn_data_baseline, col)
    except Exception as e:
        print(f"Could not plot {col}: {e}")

save_column_pair_plot(folder, real_data, syn_data_baseline, ['lactate_max', 'lactate_min'])


c:\Users\maria\Code\master\env\dpcgan_gpu\lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [ ]:
quality_report.get_visualization('Column Pair Trends').write_html(os.path.join(folder, f"quality_pair_trends.html"))

In [33]:
quality_report.get_visualization('Column Shapes')

### Real VS Baseline without race

In [ ]:
real_data_no_race = real_data.drop("race", axis=1)
syn_data_baseline_no_race = pd.read_csv("../dp_cgans/tests/results/output/syn_data_shap_baseline_norace.csv")
syn_data_baseline_no_race= syn_data_baseline_no_race.drop(syn_data_baseline_no_race.columns[0], axis=1)
metadata = get_submetadata(SingleTableMetadata.load_from_json("icu_dka_metadata.json").to_dict(), real_data_no_race.columns)

In [ ]:
diagnostic_report, quality_report, folder = evaluate_synthetic(
    exp_name="baseline_shap_no_race",
    real_data=real_data_no_race,
    syn_data=syn_data_baseline_no_race,
    metadata=metadata,
    out_dir="results_sdmetrics"
)

Generating report ...




(1/3) Evaluating Coverage: : 100%|██████████| 30/30 [00:00<00:00, 940.02it/s]




(2/3) Evaluating Boundary: : 100%|██████████| 30/30 [00:00<00:00, 501.46it/s]




(2/2) Evaluating Column Pair Trends: :   4%|▍         | 18/465 [02:04<51:31,  6.92s/it]


(3/3) Evaluating Synthesis: : 100%|██████████| 1/1 [00:56<00:00, 56.56s/it]

Diagnostic Results:

SUCCESS:
✓ The synthetic data covers over 90% of the categories present in the real data
✓ The synthetic data covers over 90% of the numerical ranges present in the real data
✓ The synthetic data follows over 90% of the min/max boundaries set by the real data
✓ Over 90% of the synthetic rows are not copies of the real data
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 435/435 [00:04<00:00, 104.86it/s]

Overall Quality Score: 87.49%

Properties:
- Column Shapes: 84.51%
- Column Pair Trends: 90.47%
New Row Synthesis Score: 1.0


In [ ]:
for col in ["age", "lactate_max", "sofa"]:
    try:
        save_column_plot(folder, real_data_no_race, syn_data_baseline_no_race, col)
    except Exception as e:
        print(f"Could not plot {col}: {e}")

save_column_pair_plot(folder, real_data_no_race, syn_data_baseline_no_race, ['lactate_max', 'lactate_min'])

In [45]:
quality_report.get_visualization('Column Pair Trends').write_html(os.path.join(folder, f"quality_pair_trends.html"))
quality_report.get_visualization('Column Pair Trends')

### Real VS Synthetic 10 SHAP

In [48]:
syn_data_10 = pd.read_csv("../dp_cgans/tests/results/output/syn_data_shap_10.csv")
syn_data_10= syn_data_10.drop(syn_data_10.columns[0], axis=1)
metadata = get_submetadata(SingleTableMetadata.load_from_json("icu_dka_metadata.json").to_dict(), real_data.columns)

In [49]:
diagnostic_report, quality_report, folder = evaluate_synthetic(
    exp_name="syn_shap_10",
    real_data=real_data,
    syn_data=syn_data_10,
    metadata=metadata,
    out_dir="results_sdmetrics"
)

Generating report ...
(3/3) Evaluating Synthesis: : 100%|██████████| 1/1 [01:14<00:00, 74.05s/it]

Diagnostic Results:

SUCCESS:
✓ The synthetic data covers over 90% of the numerical ranges present in the real data
✓ The synthetic data follows over 90% of the min/max boundaries set by the real data
✓ Over 90% of the synthetic rows are not copies of the real data

! The synthetic data is missing more than 10% of the categories present in the real data
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 465/465 [00:04<00:00, 113.50it/s]

Overall Quality Score: 88.7%

Properties:
- Column Shapes: 88.45%
- Column Pair Trends: 88.95%
New Row Synthesis Score: 1.0


In [50]:
for col in ["age", "lactate_max", "sofa"]:
    try:
        save_column_plot(folder, real_data, syn_data_10, col)
    except Exception as e:
        print(f"Could not plot {col}: {e}")

save_column_pair_plot(folder, real_data, syn_data_10, ['lactate_max', 'lactate_min'])

In [51]:
quality_report.get_visualization('Column Pair Trends').write_html(os.path.join(folder, f"quality_pair_trends.html"))
quality_report.get_visualization('Column Pair Trends')

### Real VS Synthetic 5 SHAP

In [14]:
syn_data_5 = pd.read_csv("../dp_cgans/tests/results/output/syn_data_shap_5.csv")
syn_data_5= syn_data_5.drop(syn_data_5.columns[0], axis=1)
metadata = get_submetadata(SingleTableMetadata.load_from_json("icu_dka_metadata.json").to_dict(), real_data.columns)

diagnostic_report, quality_report, folder = evaluate_synthetic(
    exp_name="syn_shap_5",
    real_data=real_data,
    syn_data=syn_data_5,
    metadata=metadata,
    out_dir="results_sdmetrics"
)

# print("Distribution similarity")
# # Numerical columns
# # age, lactate_max, sofa
# # Kolmogorov–Smirnov (KS) Test
# ks = KSStatistic.compute(real_data['age'], syn_data_5['age'])
# print(f"KS Statistic for age: {ks}")
# # Kullback–Leibler (KL) Divergence
# kl = KLDivergence.compute(real_data['age'], syn_data_5['age'])
# print(f"KL Divergence for age: {kl}")
# Wasserstein Distance TODO
# Categorical columns
# race
# Chi-Squared Test
TVComplement.compute(real_data['race'], syn_data_5['race'])
print(f"TV Complement for race: {TVComplement.compute(real_data['race'], syn_data_5['race'])}") 
# # KL
# kl = KLDivergence.compute(real_data['race'], syn_data_5['race'])
# print(f"KL Divergence for race: {kl}")  
# print("Correlation similarity")
# corr_score = CorrelationSimilarity.compute(
#     real_data=real_data,
#     synthetic_data=syn_data_5,
#     metadata=metadata
# )
# print(f"Correlation Similarity Score: {corr_score}")


for col in ["age", "lactate_max", "sofa"]:
    try:
        save_column_plot(folder, real_data, syn_data_5, col)
    except Exception as e:
        print(f"Could not plot {col}: {e}")

save_column_pair_plot(folder, real_data, syn_data_5, ['lactate_max', 'lactate_min'])
quality_report.get_visualization('Column Pair Trends').write_html(os.path.join(folder, f"quality_pair_trends.html"))
quality_report.get_visualization('Column Pair Trends')

Generating report ...
(3/3) Evaluating Synthesis: : 100%|██████████| 1/1 [01:41<00:00, 101.84s/it]

Diagnostic Results:

SUCCESS:
✓ The synthetic data covers over 90% of the numerical ranges present in the real data
✓ The synthetic data follows over 90% of the min/max boundaries set by the real data
✓ Over 90% of the synthetic rows are not copies of the real data

! The synthetic data is missing more than 10% of the categories present in the real data
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 465/465 [00:03<00:00, 117.42it/s]

Overall Quality Score: 88.69%

Properties:
- Column Shapes: 88.29%
- Column Pair Trends: 89.09%
New Row Synthesis Score: 1.0
TV Complement for race: 0.7097390197326543


### Real VS Synthetic 10 SHAP per patient

In [13]:

syn_data = pd.read_csv("../dp_cgans/tests/results/output/2025_11_24_11_57_45_syn_data_shap_10_per_patient.csv")
syn_data= syn_data.drop(syn_data.columns[0], axis=1)
metadata = get_submetadata(SingleTableMetadata.load_from_json("icu_dka_metadata.json").to_dict(), real_data.columns)

diagnostic_report, quality_report, folder = evaluate_synthetic(
    exp_name="syn_shap_10_pp",
    real_data=real_data,
    syn_data=syn_data,
    metadata=metadata,
    out_dir="results_sdmetrics"
)

for col in ["age", "lactate_max", "sofa"]:
    try:
        save_column_plot(folder, real_data, syn_data, col)
    except Exception as e:
        print(f"Could not plot {col}: {e}")

save_column_pair_plot(folder, real_data, syn_data, ['lactate_max', 'lactate_min'])
quality_report.get_visualization('Column Pair Trends').write_html(os.path.join(folder, f"quality_pair_trends.html"))
quality_report.get_visualization('Column Pair Trends')

Generating report ...
(3/3) Evaluating Synthesis: : 100%|██████████| 1/1 [01:06<00:00, 66.85s/it]

Diagnostic Results:

SUCCESS:
✓ The synthetic data covers over 90% of the numerical ranges present in the real data
✓ The synthetic data follows over 90% of the min/max boundaries set by the real data
✓ Over 90% of the synthetic rows are not copies of the real data

! The synthetic data is missing more than 10% of the categories present in the real data
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 465/465 [00:06<00:00, 76.44it/s] 

Overall Quality Score: 89.18%

Properties:
- Column Shapes: 89.43%
- Column Pair Trends: 88.93%
New Row Synthesis Score: 1.0
